<a href="https://colab.research.google.com/github/ronggobp/Machine-Learning-Course-2026/blob/main/notebooks/week-05-ensemble-learning/05_Advanced_Ensemble_Fraud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Setup & Load Imbalanced Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, average_precision_score, confusion_matrix
from sklearn.datasets import fetch_openml
from imblearn.over_sampling import SMOTE
import gradio as gr
import joblib
import wandb

# Inisialisasi W&B untuk tracking eksperimen
run = wandb.init(project="credit-card-fraud-detection", name="xgboost-smote-real-imbalance")

print("Sedang mengambil dataset asli (imbalanced) dari OpenML...")
# ID 1597 adalah dataset Credit Card Fraud asli yang sangat timpang (284.807 baris)
data = fetch_openml(data_id=1597, as_frame=True, parser='auto')

# Menggabungkan data
df = pd.concat([data.data, data.target], axis=1)
df.columns = [*df.columns[:-1], 'Class']
df['Class'] = df['Class'].astype(int)

# Untuk efisiensi waktu kelas, kita ambil sample 50.000 data namun tetap menjaga ketimpangannya
df = df.sample(n=50000, random_state=42)

print(f"Dataset Berhasil Dimuat! Shape: {df.shape}")
print("\nProporsi Kelas Asli (SANGAT TIMPANG):")
print(df['Class'].value_counts()) # Lihat angka aslinya
print(df['Class'].value_counts(normalize=True)) # Lihat persentasenya

# Log sampel ke W&B
wandb.log({"raw_sample": wandb.Table(dataframe=df.head(100))})

Visualisasi Masalah Imbalance

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='Class', data=df, palette='magma', hue='Class', legend=False)
plt.title("Distribusi Transaksi (Skala Log)\n0: Normal | 1: Fraud")
plt.yscale('log') # Skala log digunakan agar kelas minoritas (Fraud) terlihat jelas
plt.show()

Handling Imbalance dengan SMOTE

In [ ]:
X = df.drop(columns=['Class'])
y = df['Class']

# Menggunakan stratify agar proporsi kelas di data latih dan uji tetap sama
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("\nMenjalankan SMOTE untuk menyeimbangkan data...")
# SMOTE menciptakan data 'Fraud' sintetis agar AI belajar pola penipuan
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"Jumlah data setelah SMOTE: {len(y_train_res)}")
print(f"Proporsi kelas setelah SMOTE (Sudah Seimbang 50:50): \n{pd.Series(y_train_res).value_counts(normalize=True)}")

Visualisasi Efek SMOTE

In [ ]:
# Bandingkan jumlah data sebelum vs sesudah SMOTE secara visual
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
y_train.value_counts().plot(kind='bar', color=['blue', 'red'])
plt.title("Sebelum SMOTE (Sangat Timpang)")

plt.subplot(1, 2, 2)
pd.Series(y_train_res).value_counts().plot(kind='bar', color=['blue', 'red'])
plt.title("Sesudah SMOTE (Sudah Seimbang)")
plt.show()

Training XGBoost & Evaluasi AUPRC

In [ ]:
# XGBoost adalah algoritma 'Boosting' yang memperbaiki kesalahan secara bertahap
xgb_model = XGBClassifier(eval_metric='logloss', random_state=42)
xgb_model.fit(X_train_res, y_train_res)

# Prediksi probabilitas
y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]

# AUPRC adalah metrik wajib untuk data tidak seimbang (fokus pada penemuan Fraud)
auprc = average_precision_score(y_test, y_prob)

wandb.log({
    "AUPRC_Score": auprc,
    "Accuracy_Score": xgb_model.score(X_test, y_test)
})

print(f"\n--- HASIL EVALUASI XGBOOST ---")
print(f"AUPRC Score: {auprc:.4f}")
print("\nClassification Report (Perhatikan Recall pada kelas 1):")
print(classification_report(y_test, y_pred))

Visualisasi Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['Ditebak Normal', 'Ditebak Fraud'],
            yticklabels=['Asli Normal', 'Asli Fraud'])
plt.title("Confusion Matrix: Fraud Detection (XGBoost + SMOTE)")
plt.ylabel("Kondisi Aktual")
plt.xlabel("Tebakan Model")
plt.show()

wandb.finish()

In [ ]:
# Simpan model penipuan yang benar
joblib.dump(xgb_model, 'fraud_detection_model.pkl')
print("Model Berhasil Disimpan!")

# Load model
model = joblib.load('fraud_detection_model.pkl')

# Ambil rata-rata fitur V1-V28 untuk placeholder input yang tidak dimasukkan user
feature_means = X_train.mean().values

def predict_fraud_fixed(time_input, amount, v1_value, v2_value):
    try:
        # Menyiapkan array sesuai ukuran asli model (29 fitur)
        input_data = feature_means.copy()

        # Jika size adalah 29, maka indeks terakhir adalah 28
        # Kita asumsikan urutannya adalah V1, V2, ..., V28, Amount
        input_data[0] = v1_value   # Indeks 0 untuk V1
        input_data[1] = v2_value   # Indeks 1 untuk V2
        input_data[-1] = amount    # Indeks terakhir untuk Amount (index 28)

        # Catatan: Jika error tetap muncul, kolom 'Time' mungkin tidak digunakan oleh model Anda

        features = input_data.reshape(1, -1)
        prediction = model.predict(features)[0]
        probability = model.predict_proba(features)[0][1]

        if prediction == 1:
            return f"🚨 TERDETEKSI FRAUD! (Probabilitas: {probability:.2%})"
        else:
            return f"✅ TRANSAKSI AMAN (Probabilitas Fraud: {probability:.2%})"

    except Exception as e:
        return f"❌ Kesalahan sistem: {str(e)}"

# Update Interface
demo = gr.Interface(
    fn=predict_fraud_fixed,
    inputs=[
        gr.Number(label="Waktu (Bisa diisi 0 jika tidak digunakan)"),
        gr.Number(label="Jumlah Transaksi (Amount)"),
        gr.Slider(-50, 50, label="Indikator PCA V1"),
        gr.Slider(-50, 50, label="Indikator PCA V2")
    ],
    outputs="text",
    title="💳 Sistem Deteksi Penipuan Kartu Kredit"
)
demo.launch()